# 03 — Exploratory Data Analysis

**Input:** `data/processed/hour_clean.csv`

**Goals:**
- Understand the distribution of the target (`cnt`)
- Uncover temporal demand patterns (hour, weekday, month, season)
- Quantify weather impact on rentals
- Correlation analysis between features and target
- Formulate hypotheses for the modelling notebooks

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 30)

In [ ]:
df = pd.read_csv("../data/processed/hour_clean.csv", parse_dates=["dteday"])
df.head()

## 1. Target distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df["cnt"].plot(kind="hist", bins=50, ax=axes[0], title="cnt distribution")
np.log1p(df["cnt"]).plot(kind="hist", bins=50, ax=axes[1], title="log1p(cnt) distribution")
plt.tight_layout()

In [ ]:
# TODO: compute skewness of cnt and log1p(cnt)
# Question: should we log-transform the target for modelling?
print(f"cnt skew:        {df['cnt'].skew():.3f}")
print(f"log1p(cnt) skew: {np.log1p(df['cnt']).skew():.3f}")

## 2. Temporal patterns

In [ ]:
# Hourly pattern — split by workingday vs weekend
fig, ax = plt.subplots(figsize=(12, 5))
for label, group in df.groupby("workingday"):
    group.groupby("hr")["cnt"].mean().plot(ax=ax, label="Working day" if label else "Weekend/Holiday")
ax.set(title="Average rentals by hour", xlabel="Hour", ylabel="Mean cnt")
ax.legend()
plt.tight_layout()

In [ ]:
# TODO: plot monthly and seasonal demand
# Hint: groupby mnth and season, compute mean cnt
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Monthly
df.groupby("mnth")["cnt"].mean().plot(kind="bar", ax=axes[0], title="Mean rentals by month")

# Seasonal
season_map = {1: "Spring", 2: "Summer", 3: "Fall", 4: "Winter"}
df["season_label"] = df["season"].map(season_map)
df.groupby("season_label")["cnt"].mean().reindex(["Spring","Summer","Fall","Winter"]).plot(
    kind="bar", ax=axes[1], title="Mean rentals by season"
)
plt.tight_layout()

In [ ]:
# Heatmap: hour × weekday
pivot = df.pivot_table(values="cnt", index="hr", columns="weekday", aggfunc="mean")
plt.figure(figsize=(10, 8))
sns.heatmap(pivot, cmap="YlOrRd", linewidths=0.3)
plt.title("Mean rentals — hour vs weekday")
plt.tight_layout()

## 3. Weather impact

In [ ]:
# TODO: boxplots of cnt by weathersit
weather_map = {1: "Clear", 2: "Mist", 3: "Light rain/snow", 4: "Heavy rain"}
df["weather_label"] = df["weathersit"].map(weather_map)
order = ["Clear", "Mist", "Light rain/snow", "Heavy rain"]

plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x="weather_label", y="cnt", order=order)
plt.title("Rental distribution by weather")
plt.tight_layout()

In [ ]:
# Scatter: temp vs cnt, coloured by season
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, col in zip(axes, ["temp", "hum"]):
    scatter = ax.scatter(df[col], df["cnt"], c=df["season"], alpha=0.15, cmap="tab10", s=8)
    ax.set(xlabel=col, ylabel="cnt", title=f"{col} vs cnt")
plt.tight_layout()

## 4. Correlation analysis

In [ ]:
# TODO: compute and plot correlation heatmap for numerical features
num_cols = ["temp", "atemp", "hum", "windspeed", "hr", "mnth", "cnt"]
corr = df[num_cols].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation matrix")
plt.tight_layout()

In [ ]:
# Point-biserial correlations with binary features
binary_cols = ["holiday", "workingday", "is_weekend"] if "is_weekend" in df.columns else ["holiday", "workingday"]
for col in binary_cols:
    if col in df.columns:
        r = df[[col, "cnt"]].corr().iloc[0, 1]
        print(f"{col:15s} | r = {r:.3f}")

## 5. Key insights (fill in as you explore)

- **Temporal:** ...
- **Weather:** ...
- **Target distribution:** ...
- **Hypotheses for modelling:** ...